In [2]:

import carla
import cv2
import numpy as np
import queue
import random


# ============================================================
# Configuration
# ============================================================

HOST = "127.0.0.1"
PORT = 2000
TM_PORT = 8000

IMAGE_WIDTH = 800
IMAGE_HEIGHT = 600

NUM_TRAFFIC_VEHICLES = 25
NUM_PEDESTRIANS = 10

FIXED_DELTA_SECONDS = 0.05  # 20 simulation frames per second


# ============================================================
# Camera utilities
# ============================================================

def set_camera_resolution(blueprint):
    blueprint.set_attribute("image_size_x", str(IMAGE_WIDTH))
    blueprint.set_attribute("image_size_y", str(IMAGE_HEIGHT))
    blueprint.set_attribute("fov", "90")


def camera_callback(image_queue):
    """Return a callback that stores raw CARLA camera frames."""
    def callback(image):
        image_queue.put(image)

    return callback


def get_frame(image_queue, target_frame):
    """Retrieve the image corresponding to a specific simulation frame."""
    while True:
        image = image_queue.get(timeout=3.0)

        if image.frame == target_frame:
            return image

        if image.frame > target_frame:
            raise RuntimeError(
                f"Missed camera frame {target_frame}; "
                f"received frame {image.frame}."
            )


def image_to_bgr(image):
    """Convert CARLA's BGRA buffer to an OpenCV BGR image."""
    bgra = np.frombuffer(image.raw_data, dtype=np.uint8)
    bgra = bgra.reshape((image.height, image.width, 4))
    return bgra[:, :, :3].copy()


def add_label(image, label):
    """Add a title to a camera image."""
    output = image.copy()

    cv2.putText(
        output,
        label,
        (15, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    return output


# ============================================================
# Main simulation
# ============================================================

client = carla.Client(HOST, PORT)
client.set_timeout(10.0)

world = client.get_world()
blueprints = world.get_blueprint_library()
traffic_manager = client.get_trafficmanager(TM_PORT)

# Save the original world settings so they can be restored.
original_settings = world.get_settings()

vehicle = None
sensors = []
traffic_vehicles = []
pedestrians = []
pedestrian_controllers = []

rgb_queue = queue.Queue()
depth_queue = queue.Queue()
instance_queue = queue.Queue()

synchronous_enabled = False
traffic_manager_sync_enabled = False


try:
    # --------------------------------------------------------
    # 1. Enable synchronous simulation
    # --------------------------------------------------------

    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = FIXED_DELTA_SECONDS

    world.apply_settings(settings)
    synchronous_enabled = True

    traffic_manager.set_synchronous_mode(True)
    traffic_manager_sync_enabled = True

    print("Synchronous simulation enabled.")

    # --------------------------------------------------------
    # 2. Spawn the ego vehicle
    # --------------------------------------------------------

    spawn_points = world.get_map().get_spawn_points()
    random.shuffle(spawn_points)

    # Prefer the Cybertruck; fall back to another vehicle
    # if this blueprint is unavailable.
    try:
        ego_blueprint = blueprints.find("vehicle.tesla.cybertruck")
    except Exception:
        ego_blueprint = blueprints.filter("vehicle.*")[0]

    for spawn_point in spawn_points:
        vehicle = world.try_spawn_actor(
            ego_blueprint,
            spawn_point,
        )

        if vehicle is not None:
            break

    if vehicle is None:
        raise RuntimeError(
            "Could not spawn the ego vehicle. "
            "Try another map or ensure a spawn point is available."
        )

    print(f"Ego vehicle spawned: {vehicle.type_id}")

    # --------------------------------------------------------
    # 3. Spawn traffic vehicles
    # --------------------------------------------------------

    vehicle_blueprints = blueprints.filter("vehicle.*")

    # Exclude bicycles and motorcycles where the blueprint
    # exposes the number of wheels.
    four_wheel_blueprints = []

    for blueprint in vehicle_blueprints:
        if blueprint.has_attribute("number_of_wheels"):
            if int(blueprint.get_attribute("number_of_wheels")) == 4:
                four_wheel_blueprints.append(blueprint)

    if four_wheel_blueprints:
        vehicle_blueprints = four_wheel_blueprints

    traffic_spawn_points = world.get_map().get_spawn_points()
    random.shuffle(traffic_spawn_points)

    ego_location = vehicle.get_location()

    for spawn_point in traffic_spawn_points:
        if len(traffic_vehicles) >= NUM_TRAFFIC_VEHICLES:
            break

        # Keep NPC vehicles away from the ego vehicle at startup.
        if spawn_point.location.distance(ego_location) < 12.0:
            continue

        blueprint = random.choice(vehicle_blueprints)

        npc_vehicle = world.try_spawn_actor(
            blueprint,
            spawn_point,
        )

        if npc_vehicle is None:
            continue

        npc_vehicle.set_autopilot(True, TM_PORT)
        traffic_vehicles.append(npc_vehicle)

    print(f"Traffic vehicles spawned: {len(traffic_vehicles)}")

    # --------------------------------------------------------
    # 4. Spawn pedestrians and AI controllers
    # --------------------------------------------------------

    walker_blueprints = blueprints.filter("walker.pedestrian.*")
    controller_blueprint = blueprints.find("controller.ai.walker")

    for _ in range(NUM_PEDESTRIANS):
        location = world.get_random_location_from_navigation()

        if location is None:
            continue

        walker_blueprint = random.choice(walker_blueprints)

        walker = world.try_spawn_actor(
            walker_blueprint,
            carla.Transform(location),
        )

        if walker is None:
            continue

        controller = world.try_spawn_actor(
            controller_blueprint,
            carla.Transform(),
            attach_to=walker,
        )

        if controller is None:
            walker.destroy()
            continue

        pedestrians.append(walker)
        pedestrian_controllers.append(controller)

    # Start controllers after spawning walkers.
    # Tick the world to initialize actors before issuing commands.
    world.tick()

    for controller in pedestrian_controllers:
        controller.start()
        controller.set_max_speed(random.uniform(0.8, 1.8))

        destination = world.get_random_location_from_navigation()

        if destination is not None:
            controller.go_to_location(destination)

    print(f"Pedestrians spawned: {len(pedestrians)}")

    # --------------------------------------------------------
    # 5. Enable autonomous driving for the ego vehicle
    # --------------------------------------------------------

    vehicle.set_autopilot(True, TM_PORT)

    print("Ego vehicle autopilot enabled.")

    # --------------------------------------------------------
    # 6. Create three cameras
    # --------------------------------------------------------

    camera_transform = carla.Transform(
        carla.Location(x=0.0, z=3.0)
    )

    # RGB camera
    rgb_blueprint = blueprints.find("sensor.camera.rgb")
    set_camera_resolution(rgb_blueprint)

    rgb_camera = world.spawn_actor(
        rgb_blueprint,
        camera_transform,
        attach_to=vehicle,
    )
    sensors.append(rgb_camera)

    # Depth camera
    depth_blueprint = blueprints.find("sensor.camera.depth")
    set_camera_resolution(depth_blueprint)

    depth_camera = world.spawn_actor(
        depth_blueprint,
        camera_transform,
        attach_to=vehicle,
    )
    sensors.append(depth_camera)

    # Instance segmentation camera
    instance_blueprint = blueprints.find(
        "sensor.camera.instance_segmentation"
    )
    set_camera_resolution(instance_blueprint)

    instance_camera = world.spawn_actor(
        instance_blueprint,
        camera_transform,
        attach_to=vehicle,
    )
    sensors.append(instance_camera)

    # Start all sensor streams.
    rgb_camera.listen(camera_callback(rgb_queue))
    depth_camera.listen(camera_callback(depth_queue))
    instance_camera.listen(camera_callback(instance_queue))

    print("RGB, depth, and instance-segmentation cameras started.")
    print("Press Q or ESC in the display window to stop.")

    # --------------------------------------------------------
    # 7. Run simulation and display synchronized frames
    # --------------------------------------------------------

    while True:
        # Advance the synchronous simulation by one frame.
        frame_id = world.tick()

        # Retrieve matching images from all three cameras.
        rgb_image = get_frame(rgb_queue, frame_id)
        depth_image = get_frame(depth_queue, frame_id)
        instance_image = get_frame(instance_queue, frame_id)

        # RGB image: CARLA BGRA buffer -> OpenCV BGR.
        rgb_view = image_to_bgr(rgb_image)

        # Depth visualization.
        # This is a visualization, not metric distance output.
        depth_image.convert(carla.ColorConverter.LogarithmicDepth)
        depth_view = image_to_bgr(depth_image)

        # Instance segmentation visualization.
        instance_image.convert(carla.ColorConverter.CityScapesPalette)
        instance_view = image_to_bgr(instance_image)

        # Add labels.
        rgb_view = add_label(rgb_view, "RGB")
        depth_view = add_label(depth_view, "DEPTH")
        instance_view = add_label(
            instance_view,
            "INSTANCE SEGMENTATION",
        )

        # Combine all three views horizontally.
        combined_view = np.hstack(
            (rgb_view, depth_view, instance_view)
        )

        cv2.imshow("CARLA - Camera Perception Streams", combined_view)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q") or key == 27:
            break

except KeyboardInterrupt:
    print("Simulation interrupted by user.")

finally:
    # --------------------------------------------------------
    # 8. Cleanup
    # --------------------------------------------------------

    print("Cleaning up actors and sensors...")

    # Stop cameras before destroying them.
    for sensor in sensors:
        if sensor.is_alive:
            sensor.stop()

    # Stop pedestrian controllers first.
    for controller in pedestrian_controllers:
        if controller.is_alive:
            try:
                controller.stop()
            except RuntimeError:
                pass

    # Destroy controllers, then walkers.
    for controller in pedestrian_controllers:
        if controller.is_alive:
            controller.destroy()

    for pedestrian in pedestrians:
        if pedestrian.is_alive:
            pedestrian.destroy()

    # Destroy traffic vehicles.
    for npc_vehicle in traffic_vehicles:
        if npc_vehicle.is_alive:
            npc_vehicle.destroy()

    # Destroy cameras.
    for sensor in reversed(sensors):
        if sensor.is_alive:
            sensor.destroy()

    # Destroy the ego vehicle.
    if vehicle is not None and vehicle.is_alive:
        vehicle.destroy()

    cv2.destroyAllWindows()

    # Restore Traffic Manager and world settings.
    if traffic_manager_sync_enabled:
        traffic_manager.set_synchronous_mode(False)

    if synchronous_enabled:
        world.apply_settings(original_settings)

    print("Cleanup complete.")


Synchronous simulation enabled.
Ego vehicle spawned: vehicle.tesla.cybertruck
Traffic vehicles spawned: 25
Pedestrians spawned: 10
Ego vehicle autopilot enabled.
RGB, depth, and instance-segmentation cameras started.
Press Q or ESC in the display window to stop.


: 